In [ ]:
'''
K-Means independently for each image
'''
import cv2
import numpy as np
from skimage.feature import hog, local_binary_pattern
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
import glob
import os

# ---------- Parameters ----------
GRID_ROWS = 8
GRID_COLS = 8
NUM_GRIDS = GRID_ROWS * GRID_COLS
TARGET_W, TARGET_H = 800, 600
SUBGRID_SIZE = 4  # sub-blocks per grid
LBP_PTS = 8
LBP_RADIUS = 1
HOG_PIXELS_PER_CELL = (8, 8)
NUM_CLUSTERS = 2

# ---------- Preprocessing ----------
def enforce_4_3_aspect_and_scale(img):
    h, w = img.shape[:2]
    desired_ratio = 4/3
    cur_ratio = w / h
    if abs(cur_ratio - desired_ratio) > 1e-3:
        if cur_ratio > desired_ratio:
            new_w = int(h * desired_ratio)
            x0 = (w - new_w) // 2
            img = img[:, x0:x0+new_w]
        else:
            new_h = int(w / desired_ratio)
            y0 = (h - new_h) // 2
            img = img[y0:y0+new_h, :]
    h2, w2 = img.shape[:2]
    scale = min(1.0, min(TARGET_W / w2, TARGET_H / h2))
    if scale < 1.0:
        new_w = int(w2 * scale)
        new_h = int(h2 * scale)
        img = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)
    pad_w = TARGET_W - img.shape[1]
    pad_h = TARGET_H - img.shape[0]
    top = pad_h//2 if pad_h>0 else 0
    bottom = pad_h-top if pad_h>0 else 0
    left = pad_w//2 if pad_w>0 else 0
    right = pad_w-left if pad_w>0 else 0
    if pad_w>0 or pad_h>0:
        img = cv2.copyMakeBorder(img, top, bottom, left, right, cv2.BORDER_REFLECT)
    return img

# ---------- Grid splitting ----------
def split_to_grids(img, rows=GRID_ROWS, cols=GRID_COLS):
    h, w = img.shape[:2]
    cell_h = h // rows
    cell_w = w // cols
    grids = []
    for r in range(rows):
        for c in range(cols):
            y0 = r * cell_h
            x0 = c * cell_w
            y1 = h if r==rows-1 else y0+cell_h
            x1 = w if c==cols-1 else x0+cell_w
            grids.append(img[y0:y1, x0:x1])
    return grids

# ---------- Feature extraction ----------
def extract_features_for_cell(cell):
    gray = cv2.cvtColor(cell, cv2.COLOR_BGR2GRAY)
    
    # ---------- Existing features ----------
    # HOG
    hog_feat = hog(gray, pixels_per_cell=HOG_PIXELS_PER_CELL, cells_per_block=(2,2), feature_vector=True)
    
    # LBP histogram
    lbp = local_binary_pattern(gray, P=LBP_PTS, R=LBP_RADIUS, method='uniform')
    hist_lbp, _ = np.histogram(lbp.ravel(), bins=np.arange(0, LBP_PTS+3), range=(0, LBP_PTS+2))
    hist_lbp = hist_lbp.astype("float") / (hist_lbp.sum() + 1e-6)
    
    # HSV histogram (H + S)
    hsv = cv2.cvtColor(cell, cv2.COLOR_BGR2HSV)
    h_hist = cv2.calcHist([hsv], [0], None, [16], [0,180]).flatten()
    s_hist = cv2.calcHist([hsv], [1], None, [8], [0,256]).flatten()
    color_hist = np.concatenate([h_hist, s_hist]).astype("float")
    color_hist /= (color_hist.sum() + 1e-6)
    
    # ORB keypoint count
    orb = cv2.ORB_create(nfeatures=200)
    kps = orb.detect(gray, None)
    kp_count = np.array([len(kps)], dtype=float)
    
    # Edge density
    edges = cv2.Canny(gray,50,150)
    edge_density = np.array([edges.mean()], dtype=float)
    
    # Saliency (spectral residual)
    try:
        saliency = cv2.saliency.StaticSaliencySpectralResidual_create()
        success, sal_map = saliency.computeSaliency(cell)
        if success:
            sal_frac = np.array([(sal_map>0.5).mean()], dtype=float)
        else:
            sal_frac = np.array([0.0], dtype=float)
    except AttributeError:
        sal_frac = np.array([0.0], dtype=float)
    
    # ---------- New Filters ----------
    # 1. Sobel (gradient in x and y)
    sobel_x = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    sobel_y = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
    sobel_mag = np.sqrt(sobel_x**2 + sobel_y**2)
    sobel_feat = np.array([sobel_mag.mean(), sobel_mag.std()], dtype=float)
    
    # 2. Laplacian (second derivative / blob detection)
    lap = cv2.Laplacian(gray, cv2.CV_64F)
    lap_feat = np.array([lap.mean(), lap.std()], dtype=float)
    
    # 3. Gabor filters (multi-orientation)
    gabor_feats = []
    # Use 4 orientations
    for theta in [0, np.pi/4, np.pi/2, 3*np.pi/4]:
        kernel = cv2.getGaborKernel((7,7), sigma=3, theta=theta, lambd=10, gamma=0.5, psi=0)
        filtered = cv2.filter2D(gray, cv2.CV_64F, kernel)
        gabor_feats.extend([filtered.mean(), filtered.std()])
    gabor_feats = np.array(gabor_feats, dtype=float)
    
    # ---------- Combine all features ----------
    feat = np.concatenate([
        hog_feat,
        hist_lbp,
        color_hist,
        kp_count,
        edge_density,
        sal_frac,
        sobel_feat,
        lap_feat,
        gabor_feats
    ])
    
    return feat


def extract_features_for_subblocks(cell, subgrid_size=SUBGRID_SIZE):
    h, w = cell.shape[:2]
    sub_h, sub_w = h//subgrid_size, w//subgrid_size
    sub_features = []
    for i in range(subgrid_size):
        for j in range(subgrid_size):
            subblock = cell[i*sub_h:(i+1)*sub_h, j*sub_w:(j+1)*sub_w]
            sub_features.append(extract_features_for_cell(subblock))
    return np.array(sub_features)

# ---------- Image processing ----------
def process_image(img):
    grids = split_to_grids(img)
    all_sub_features = []
    cell_indices = []
    for idx, cell in enumerate(grids):
        sub_feats = extract_features_for_subblocks(cell)
        all_sub_features.extend(sub_feats)
        cell_indices.extend([idx]*len(sub_feats))
    return np.array(all_sub_features), np.array(cell_indices)

def highlight_cells(img, labels):
    h, w = img.shape[:2]
    cell_h, cell_w = h//GRID_ROWS, w//GRID_COLS
    img_copy = img.copy()
    for i in range(GRID_ROWS):
        for j in range(GRID_COLS):
            idx = i*GRID_COLS + j
            color = (0,0,255) if labels[idx]==1 else (255,255,255)
            thickness = 2 if labels[idx]==1 else 1
            cv2.rectangle(img_copy, (j*cell_w,i*cell_h), ((j+1)*cell_w,(i+1)*cell_h), color, thickness)
    return img_copy

# ---------- Main ----------
folder_path = "images"
image_paths = sorted(glob.glob(folder_path + "/*.jpg"))[:4]
print(f"Processing images: {image_paths}")

for img_path in image_paths:
    img = cv2.imread(img_path)
    img2 = enforce_4_3_aspect_and_scale(img)
    
    sub_features, cell_indices = process_image(img2)
    
    # KMeans clustering on sub-block features
    kmeans = KMeans(n_clusters=NUM_CLUSTERS, random_state=42)
    sub_labels = kmeans.fit_predict(sub_features)
    
    # Majority vote per grid
    labels = []
    for c in range(NUM_GRIDS):
        cell_subs = sub_labels[cell_indices==c]
        majority_label = np.bincount(cell_subs).argmax()
        labels.append(majority_label)
    labels = np.array(labels)
    
    # Highlight grids
    img_highlight = highlight_cells(img2, labels)
    plt.figure(figsize=(10,8))
    plt.imshow(cv2.cvtColor(img_highlight, cv2.COLOR_BGR2RGB))
    plt.title(f"Highlighted grids: {os.path.basename(img_path)}")
    plt.axis('off')
    plt.show()

In [ ]:
import cv2, sys
print("Python executable:", sys.executable)
print("cv2 file loaded from:", cv2.__file__)
print("OpenCV version:", cv2.__version__)
print("Has saliency module?", hasattr(cv2, "saliency"))


In [ ]:
import cv2
import numpy as np
import glob, os
from skimage.feature import local_binary_pattern, hog
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

# ---------- Parameters ----------
GRID_ROWS = 8
GRID_COLS = 8
NUM_GRIDS = GRID_ROWS * GRID_COLS
TARGET_W, TARGET_H = 800, 600
SUBGRID_SIZE = 4  # sub-blocks per grid
LBP_PTS = 8
LBP_RADIUS = 1
HOG_PIXELS_PER_CELL = (8, 8)
NUM_CLUSTERS = 2

# ---------- Preprocessing ----------
def enforce_4_3_aspect_and_scale(img):
    h, w = img.shape[:2]
    desired_ratio = 4/3
    cur_ratio = w / h
    if abs(cur_ratio - desired_ratio) > 1e-3:
        if cur_ratio > desired_ratio:
            new_w = int(h * desired_ratio)
            x0 = (w - new_w) // 2
            img = img[:, x0:x0+new_w]
        else:
            new_h = int(w / desired_ratio)
            y0 = (h - new_h) // 2
            img = img[y0:y0+new_h, :]
    h2, w2 = img.shape[:2]
    scale = min(1.0, min(TARGET_W / w2, TARGET_H / h2))
    if scale < 1.0:
        new_w = int(w2 * scale)
        new_h = int(h2 * scale)
        img = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)
    pad_w = TARGET_W - img.shape[1]
    pad_h = TARGET_H - img.shape[0]
    top = pad_h//2 if pad_h>0 else 0
    bottom = pad_h-top if pad_h>0 else 0
    left = pad_w//2 if pad_w>0 else 0
    right = pad_w-left if pad_w>0 else 0
    if pad_w>0 or pad_h>0:
        img = cv2.copyMakeBorder(img, top, bottom, left, right, cv2.BORDER_REFLECT)
    return img

# ---------- Grid splitting ----------
def split_to_grids(img, rows=GRID_ROWS, cols=GRID_COLS):
    h, w = img.shape[:2]
    cell_h = h // rows
    cell_w = w // cols
    grids = []
    for r in range(rows):
        for c in range(cols):
            y0 = r * cell_h
            x0 = c * cell_w
            y1 = h if r==rows-1 else y0+cell_h
            x1 = w if c==cols-1 else x0+cell_w
            grids.append(img[y0:y1, x0:x1])
    return grids

# ---------- Feature extraction ----------
def extract_features_for_cell(cell):
    gray = cv2.cvtColor(cell, cv2.COLOR_BGR2GRAY)
    
    # ---------- Existing features ----------
    # HOG
    hog_feat = hog(gray, pixels_per_cell=HOG_PIXELS_PER_CELL, cells_per_block=(2,2), feature_vector=True)
    
    # LBP histogram
    lbp = local_binary_pattern(gray, P=LBP_PTS, R=LBP_RADIUS, method='uniform')
    hist_lbp, _ = np.histogram(lbp.ravel(), bins=np.arange(0, LBP_PTS+3), range=(0, LBP_PTS+2))
    hist_lbp = hist_lbp.astype("float") / (hist_lbp.sum() + 1e-6)
    
    # HSV histogram (H + S)
    hsv = cv2.cvtColor(cell, cv2.COLOR_BGR2HSV)
    h_hist = cv2.calcHist([hsv], [0], None, [16], [0,180]).flatten()
    s_hist = cv2.calcHist([hsv], [1], None, [8], [0,256]).flatten()
    color_hist = np.concatenate([h_hist, s_hist]).astype("float")
    color_hist /= (color_hist.sum() + 1e-6)
    
    # ORB keypoint count
    orb = cv2.ORB_create(nfeatures=200)
    kps = orb.detect(gray, None)
    kp_count = np.array([len(kps)], dtype=float)
    
    # Edge density
    edges = cv2.Canny(gray,50,150)
    edge_density = np.array([edges.mean()], dtype=float)
    
    # Saliency (spectral residual)
    try:
        saliency = cv2.saliency.StaticSaliencySpectralResidual_create()
        success, sal_map = saliency.computeSaliency(cell)
        if success:
            sal_frac = np.array([(sal_map>0.5).mean()], dtype=float)
        else:
            sal_frac = np.array([0.0], dtype=float)
    except AttributeError:
        sal_frac = np.array([0.0], dtype=float)
    
    # ---------- New Filters ----------
    # 1. Sobel (gradient magnitude)
    sobel_x = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    sobel_y = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
    sobel_mag = np.sqrt(sobel_x**2 + sobel_y**2)
    sobel_feat = np.array([sobel_mag.mean(), sobel_mag.std()], dtype=float)
    
    # 2. Laplacian (second derivative / blob detection)
    lap = cv2.Laplacian(gray, cv2.CV_64F)
    lap_feat = np.array([lap.mean(), lap.std()], dtype=float)
    
    # 3. Gabor filters (multi-orientation)
    gabor_feats = []
    for theta in [0, np.pi/4, np.pi/2, 3*np.pi/4]:
        kernel = cv2.getGaborKernel((7,7), sigma=3, theta=theta, lambd=10, gamma=0.5, psi=0)
        filtered = cv2.filter2D(gray, cv2.CV_64F, kernel)
        gabor_feats.extend([filtered.mean(), filtered.std()])
    gabor_feats = np.array(gabor_feats, dtype=float)
    
    # ---------- Combine all features ----------
    feat = np.concatenate([
        hog_feat,
        hist_lbp,
        color_hist,
        kp_count,
        edge_density,
        sal_frac,
        sobel_feat,
        lap_feat,
        gabor_feats
    ])
    
    return feat

def extract_features_for_subblocks(cell, subgrid_size=SUBGRID_SIZE):
    h, w = cell.shape[:2]
    sub_h, sub_w = h//subgrid_size, w//subgrid_size
    sub_features = []

    for i in range(subgrid_size):
        for j in range(subgrid_size):
            subblock = cell[i*sub_h:(i+1)*sub_h, j*sub_w:(j+1)*sub_w]
            sub_features.append(extract_features_for_cell(subblock))
    return np.array(sub_features)

# ---------- Image processing ----------
def process_image(img):
    grids = split_to_grids(img)
    all_sub_features = []
    cell_indices = []
    for idx, cell in enumerate(grids):
        sub_feats = extract_features_for_subblocks(cell)
        all_sub_features.extend(sub_feats)
        cell_indices.extend([idx]*len(sub_feats))
    
    # Normalize features
    all_sub_features = np.array(all_sub_features)
    all_sub_features = StandardScaler().fit_transform(all_sub_features)
    
    return all_sub_features, np.array(cell_indices)

# ---------- Highlighting ----------
def highlight_cells(img, labels):
    h, w = img.shape[:2]
    cell_h, cell_w = h//GRID_ROWS, w//GRID_COLS
    img_copy = img.copy()
    for i in range(GRID_ROWS):
        for j in range(GRID_COLS):
            idx = i*GRID_COLS + j
            color = (0,0,255) if labels[idx]==1 else (255,255,255)
            thickness = 2 if labels[idx]==1 else 1
            cv2.rectangle(img_copy, (j*cell_w,i*cell_h), ((j+1)*cell_w,(i+1)*cell_h), color, thickness)
    return img_copy

# ---------- Main ----------
folder_path = "images"
image_paths = sorted(glob.glob(folder_path + "/*.jpg"))[:4]
print(f"Processing images: {image_paths}")

for img_path in image_paths:
    img = cv2.imread(img_path)
    img2 = enforce_4_3_aspect_and_scale(img)
    
    sub_features, cell_indices = process_image(img2)
    
    # KMeans clustering on sub-block features
    kmeans = KMeans(n_clusters=NUM_CLUSTERS, random_state=42)
    sub_labels = kmeans.fit_predict(sub_features)
    
    # Majority vote per grid
    labels = []
    for c in range(NUM_GRIDS):
        cell_subs = sub_labels[cell_indices==c]
        majority_label = np.bincount(cell_subs).argmax()
        labels.append(majority_label)
    labels = np.array(labels)
    
    # Highlight grids
    img_highlight = highlight_cells(img2, labels)
    plt.figure(figsize=(10,8))
    plt.imshow(cv2.cvtColor(img_highlight, cv2.COLOR_BGR2RGB))
    plt.title(f"Highlighted grids: {os.path.basename(img_path)}")
    plt.axis('off')
    plt.show()


In [ ]:
'''Lets start by considering the Gabor filter.'''
import cv2
import numpy as np
from skimage.feature import hog, local_binary_pattern
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
import glob
import os

# ---------- Parameters ----------
GRID_ROWS = 8
GRID_COLS = 8
NUM_GRIDS = GRID_ROWS * GRID_COLS
TARGET_W, TARGET_H = 800, 600
SUBGRID_SIZE = 4  # sub-blocks per grid
LBP_PTS = 8
LBP_RADIUS = 1
HOG_PIXELS_PER_CELL = (8, 8)
NUM_CLUSTERS = 2

# ---------- Preprocessing ----------
def enforce_4_3_aspect_and_scale(img):
    h, w = img.shape[:2]
    desired_ratio = 4/3
    cur_ratio = w / h
    if abs(cur_ratio - desired_ratio) > 1e-3:
        if cur_ratio > desired_ratio:
            new_w = int(h * desired_ratio)
            x0 = (w - new_w) // 2
            img = img[:, x0:x0+new_w]
        else:
            new_h = int(w / desired_ratio)
            y0 = (h - new_h) // 2
            img = img[y0:y0+new_h, :]
    h2, w2 = img.shape[:2]
    scale = min(1.0, min(TARGET_W / w2, TARGET_H / h2))
    if scale < 1.0:
        new_w = int(w2 * scale)
        new_h = int(h2 * scale)
        img = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)
    pad_w = TARGET_W - img.shape[1]
    pad_h = TARGET_H - img.shape[0]
    top = pad_h//2 if pad_h>0 else 0
    bottom = pad_h-top if pad_h>0 else 0
    left = pad_w//2 if pad_w>0 else 0
    right = pad_w-left if pad_w>0 else 0
    if pad_w>0 or pad_h>0:
        img = cv2.copyMakeBorder(img, top, bottom, left, right, cv2.BORDER_REFLECT)
    return img

# ---------- Grid splitting ----------
def split_to_grids(img, rows=GRID_ROWS, cols=GRID_COLS):
    h, w = img.shape[:2]
    cell_h = h // rows
    cell_w = w // cols
    grids = []
    for r in range(rows):
        for c in range(cols):
            y0 = r * cell_h
            x0 = c * cell_w
            y1 = h if r==rows-1 else y0+cell_h
            x1 = w if c==cols-1 else x0+cell_w
            grids.append(img[y0:y1, x0:x1])
    return grids

# ---------- Feature extraction ----------
def extract_features_for_cell(cell):
    """
    Extract Gabor features from each color channel and combine them.
    
    Parameters:
    - img: input BGR image
    - ksize: kernel size for Gabor filters
    
    Returns:
    - gabor_feats: 1D numpy array of concatenated features
    """
    gabor_feats = []

    # Split image into channels (B, G, R)
    channels = cv2.split(img)

    # Gabor parameters
    thetas = [0, np.pi/4, np.pi/2, 3*np.pi/4]
    sigmas = [1, 3]
    lambdas = [4, 8]
    gammas = [0.5, 1]

    for ch in channels:
        for theta in thetas:
            for sigma in sigmas:
                for lam in lambdas:
                    for gamma in gammas:
                        kernel = cv2.getGaborKernel(
                            ksize=(ksize, ksize),
                            sigma=sigma,
                            theta=theta,
                            lambd=lam,
                            gamma=gamma,
                            psi=0,
                            ktype=cv2.CV_32F
                        )
                        filtered = cv2.filter2D(ch, cv2.CV_8UC3, kernel)
                        gabor_feats.append(filtered.mean())
                        gabor_feats.append(filtered.var())

    return np.array(gabor_feats, dtype=float)

In [ ]:
import cv2
import numpy as np
import glob
import os
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

# ---------- Parameters ----------
GRID_ROWS = 8
GRID_COLS = 8
NUM_GRIDS = GRID_ROWS * GRID_COLS
TARGET_W, TARGET_H = 800, 600
SUBGRID_SIZE = 4  # sub-blocks per grid
NUM_CLUSTERS = 2

# Define Gabor parameters
GABOR_THETAS = np.linspace(0, np.pi, 8, endpoint=False)  # 8 orientations
GABOR_SIGMAS = [1, 2]  # scales
GABOR_LAMBDAS = [4, 8]  # wavelengths
GABOR_GAMMAS = [0.5, 1]  # aspect ratio

# HOG parameters
HOG_PIXELS_PER_CELL = (8, 8)
HOG_CELLS_PER_BLOCK = (2, 2)
HOG_ORIENTATIONS = 9

# ---------- Preprocessing ----------
def enforce_4_3_aspect_and_scale(img):
    h, w = img.shape[:2]
    desired_ratio = 4/3
    cur_ratio = w / h
    if abs(cur_ratio - desired_ratio) > 1e-3:
        if cur_ratio > desired_ratio:
            new_w = int(h * desired_ratio)
            x0 = (w - new_w) // 2
            img = img[:, x0:x0+new_w]
        else:
            new_h = int(w / desired_ratio)
            y0 = (h - new_h) // 2
            img = img[y0:y0+new_h, :]
    h2, w2 = img.shape[:2]
    scale = min(1.0, min(TARGET_W / w2, TARGET_H / h2))
    if scale < 1.0:
        new_w = int(w2 * scale)
        new_h = int(h2 * scale)
        img = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)
    pad_w = TARGET_W - img.shape[1]
    pad_h = TARGET_H - img.shape[0]
    top = pad_h//2 if pad_h>0 else 0
    bottom = pad_h-top if pad_h>0 else 0
    left = pad_w//2 if pad_w>0 else 0
    right = pad_w-left if pad_w>0 else 0
    if pad_w>0 or pad_h>0:
        img = cv2.copyMakeBorder(img, top, bottom, left, right, cv2.BORDER_REFLECT)
    return img

# ---------- Grid splitting ----------
def split_to_grids(img, rows=GRID_ROWS, cols=GRID_COLS):
    h, w = img.shape[:2]
    cell_h = h // rows
    cell_w = w // cols
    grids = []
    for r in range(rows):
        for c in range(cols):
            y0 = r * cell_h
            x0 = c * cell_w
            y1 = h if r==rows-1 else y0+cell_h
            x1 = w if c==cols-1 else x0+cell_w
            grids.append(img[y0:y1, x0:x1])
    return grids

# ---------- Gabor feature extraction ----------
def gabor_features(gray):
    """Compute Gabor features (mean, variance, skew) for a grayscale image"""
    feats = []
    for theta in GABOR_THETAS:
        for sigma in GABOR_SIGMAS:
            for lam in GABOR_LAMBDAS:
                for gamma in GABOR_GAMMAS:
                    kernel = cv2.getGaborKernel((9,9), sigma, theta, lam, gamma, 0, ktype=cv2.CV_32F)
                    filtered = cv2.filter2D(gray, cv2.CV_32F, kernel)
                    feats.append(filtered.mean())
                    feats.append(filtered.var())
                    feats.append(skew(filtered.ravel()))
    return np.array(feats, dtype=np.float32)

def extract_features_for_cell(cell):
    """
    Extract features for a cell:
    - HSV color histogram (H+S)
    - HOG descriptor
    - Gabor features
    - Saliency fraction
    """
    # --- HSV color histogram ---
    hsv = cv2.cvtColor(cell, cv2.COLOR_BGR2HSV)
    h_hist = cv2.calcHist([hsv], [0], None, [16], [0, 180]).flatten()
    s_hist = cv2.calcHist([hsv], [1], None, [8], [0, 256]).flatten()
    color_feat = np.concatenate([h_hist, s_hist])
    color_feat = color_feat / (color_feat.sum() + 1e-6)  # normalize

    # --- HOG ---
    gray = cv2.cvtColor(cell, cv2.COLOR_BGR2GRAY)
    hog_feat = hog(gray,
                   orientations=HOG_ORIENTATIONS,
                   pixels_per_cell=HOG_PIXELS_PER_CELL,
                   cells_per_block=HOG_CELLS_PER_BLOCK,
                   block_norm='L2-Hys',
                   feature_vector=True)

    # --- Gabor ---
    gabor_feat = gabor_features(gray)

    # --- Saliency fraction ---
    saliency_detector = cv2.saliency.StaticSaliencySpectralResidual_create()
    success, sal_map = saliency_detector.computeSaliency(cell)
    sal_frac = np.array([(sal_map > 128).mean()] if success else [0.0], dtype=np.float32)

    # Combine all features
    feat = np.concatenate([color_feat, hog_feat, gabor_feat, sal_frac])
    return feat

# ---------- Process image into sub-block Gabor features ----------
def process_image(img):
    grids = split_to_grids(img)
    all_sub_features = []
    cell_indices = []
    for idx, cell in enumerate(grids):
        # Split cell into sub-blocks
        h, w = cell.shape[:2]
        sub_h, sub_w = h // SUBGRID_SIZE, w // SUBGRID_SIZE
        for i in range(SUBGRID_SIZE):
            for j in range(SUBGRID_SIZE):
                subblock = cell[i*sub_h:(i+1)*sub_h, j*sub_w:(j+1)*sub_w]
                feat = extract_gabor_features_color(subblock)
                all_sub_features.append(feat)
                cell_indices.append(idx)
    return np.array(all_sub_features), np.array(cell_indices)

#Highlight cells
def highlight_cells(img, labels, alpha=0.4):
    """
    Fill grid cells labeled as 1 with semi-transparent red.
    
    Parameters:
    - img: input BGR image
    - labels: array of length NUM_GRIDS, 0 or 1
    - alpha: transparency of overlay (0=transparent, 1=opaque)
    
    Returns:
    - img_copy: image with highlighted grids
    """
    h, w = img.shape[:2]
    cell_h, cell_w = h // GRID_ROWS, w // GRID_COLS
    img_copy = img.copy()
    
    for i in range(GRID_ROWS):
        for j in range(GRID_COLS):
            idx = i * GRID_COLS + j
            if labels[idx] == 1:
                # Define the grid rectangle
                y0, y1 = i * cell_h, (i+1) * cell_h
                x0, x1 = j * cell_w, (j+1) * cell_w
                # Create red overlay
                overlay = img_copy.copy()
                overlay[y0:y1, x0:x1] = (0, 0, 255)  # BGR red
                # Blend overlay with original image
                cv2.addWeighted(overlay, alpha, img_copy, 1 - alpha, 0, img_copy)
    
    return img_copy


# ---------- Main Loop ----------
folder_path = "images"
image_paths = sorted(glob.glob(folder_path + "/*.jpg"))[:4]
print(f"Processing images: {image_paths}")

for img_path in image_paths:
    img = cv2.imread(img_path)
    img2 = enforce_4_3_aspect_and_scale(img)
    
    sub_features, cell_indices = process_image(img2)
    
    # KMeans clustering on sub-block features
    kmeans = KMeans(n_clusters=NUM_CLUSTERS, random_state=42)
    sub_labels = kmeans.fit_predict(sub_features)
    
    # Majority vote per grid
    labels = []
    for c in range(NUM_GRIDS):
        cell_subs = sub_labels[cell_indices==c]
        majority_label = np.bincount(cell_subs).argmax()
        labels.append(majority_label)
    labels = np.array(labels)
    
    # Highlight grids
    img_highlight = highlight_cells(img2, labels)
    plt.figure(figsize=(10,8))
    plt.imshow(cv2.cvtColor(img_highlight, cv2.COLOR_BGR2RGB))
    plt.title(f"Highlighted grids: {os.path.basename(img_path)}")
    plt.axis('off')
    plt.show()


In [ ]:
import cv2
import numpy as np
import glob
import os
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
from skimage.feature import local_binary_pattern
from scipy.stats import skew

# ---------- Parameters ----------
GRID_ROWS = 8
GRID_COLS = 8
NUM_GRIDS = GRID_ROWS * GRID_COLS
TARGET_W, TARGET_H = 800, 600
SUBGRID_SIZE = 4  # sub-blocks per grid
NUM_CLUSTERS = 2

# Gabor parameters
GABOR_THETAS = np.linspace(0, np.pi, 8, endpoint=False)
GABOR_SIGMAS = [1, 2]
GABOR_LAMBDAS = [4, 8]
GABOR_GAMMAS = [0.5, 1]

# ---------- Preprocessing ----------
def enforce_4_3_aspect_and_scale(img):
    h, w = img.shape[:2]
    desired_ratio = 4/3
    cur_ratio = w / h
    if abs(cur_ratio - desired_ratio) > 1e-3:
        if cur_ratio > desired_ratio:
            new_w = int(h * desired_ratio)
            x0 = (w - new_w) // 2
            img = img[:, x0:x0+new_w]
        else:
            new_h = int(w / desired_ratio)
            y0 = (h - new_h) // 2
            img = img[y0:y0+new_h, :]
    h2, w2 = img.shape[:2]
    scale = min(1.0, min(TARGET_W / w2, TARGET_H / h2))
    if scale < 1.0:
        new_w = int(w2 * scale)
        new_h = int(h2 * scale)
        img = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)
    pad_w = TARGET_W - img.shape[1]
    pad_h = TARGET_H - img.shape[0]
    top = pad_h//2 if pad_h>0 else 0
    bottom = pad_h-top if pad_h>0 else 0
    left = pad_w//2 if pad_w>0 else 0
    right = pad_w-left if pad_w>0 else 0
    if pad_w>0 or pad_h>0:
        img = cv2.copyMakeBorder(img, top, bottom, left, right, cv2.BORDER_REFLECT)
    return img

# ---------- Grid splitting ----------
def split_to_grids(img, rows=GRID_ROWS, cols=GRID_COLS):
    h, w = img.shape[:2]
    cell_h = h // rows
    cell_w = w // cols
    grids = []
    for r in range(rows):
        for c in range(cols):
            y0 = r * cell_h
            x0 = c * cell_w
            y1 = h if r == rows - 1 else y0 + cell_h
            x1 = w if c == cols - 1 else x0 + cell_w
            grids.append(img[y0:y1, x0:x1])
    return grids

# ---------- Feature extractor ----------
def extract_selected_features(cell):
    """
    Extracts Sobel, Laplacian, LBP, Gabor, HSV, and color ratio features.
    """
    gray = cv2.cvtColor(cell, cv2.COLOR_BGR2GRAY)
    hsv = cv2.cvtColor(cell, cv2.COLOR_BGR2HSV)
    b, g, r = cv2.split(cell.astype(np.float32) + 1e-6)  # avoid divide by zero
    feats = []

    # --- Sobel ---
    sobelx = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3)
    sobely = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=3)
    sobel_mag = np.sqrt(sobelx**2 + sobely**2)
    feats += [sobelx.mean(), sobely.mean(), sobel_mag.mean(),
              sobelx.var(), sobely.var(), sobel_mag.var()]

    # --- Laplacian ---
    lap = cv2.Laplacian(gray, cv2.CV_32F)
    feats += [lap.mean(), lap.var(), skew(lap.ravel())]

    # --- LBP (uniform, 8 points, radius 1) ---
    lbp = local_binary_pattern(gray, P=8, R=1, method='uniform')
    (hist, _) = np.histogram(lbp.ravel(), bins=np.arange(0, 10), range=(0, 9))
    hist = hist.astype("float")
    hist /= (hist.sum() + 1e-6)
    feats += hist.tolist()

    # --- Gabor features ---
    for theta in GABOR_THETAS:
        for sigma in GABOR_SIGMAS:
            for lam in GABOR_LAMBDAS:
                for gamma in GABOR_GAMMAS:
                    kernel = cv2.getGaborKernel((9, 9), sigma, theta, lam, gamma, 0, ktype=cv2.CV_32F)
                    filtered = cv2.filter2D(gray, cv2.CV_32F, kernel)
                    feats += [filtered.mean(), filtered.var()]

    # --- HSV stats ---
    h_mean, s_mean, v_mean = hsv[...,0].mean(), hsv[...,1].mean(), hsv[...,2].mean()
    h_std, s_std, v_std = hsv[...,0].std(), hsv[...,1].std(), hsv[...,2].std()
    feats += [h_mean, s_mean, v_mean, h_std, s_std, v_std]

    # --- Color ratios ---
    feats += [(r/g).mean(), (r/b).mean(), (g/b).mean()]

    return np.array(feats, dtype=np.float32)

# ---------- Process image ----------
def process_image(img):
    grids = split_to_grids(img)
    all_sub_features = []
    cell_indices = []

    for idx, cell in enumerate(grids):
        h, w = cell.shape[:2]
        sub_h, sub_w = h // SUBGRID_SIZE, w // SUBGRID_SIZE
        for i in range(SUBGRID_SIZE):
            for j in range(SUBGRID_SIZE):
                subblock = cell[i*sub_h:(i+1)*sub_h, j*sub_w:(j+1)*sub_w]
                feat = extract_selected_features(subblock)
                all_sub_features.append(feat)
                cell_indices.append(idx)

    return np.array(all_sub_features), np.array(cell_indices)

# ---------- Visualization ----------
def highlight_cells(img, labels, alpha=0.4):
    h, w = img.shape[:2]
    cell_h, cell_w = h // GRID_ROWS, w // GRID_COLS
    img_copy = img.copy()
    for i in range(GRID_ROWS):
        for j in range(GRID_COLS):
            idx = i * GRID_COLS + j
            if labels[idx] == 1:
                y0, y1 = i * cell_h, (i+1) * cell_h
                x0, x1 = j * cell_w, (j+1) * cell_w
                overlay = img_copy.copy()
                overlay[y0:y1, x0:x1] = (0, 0, 255)
                cv2.addWeighted(overlay, alpha, img_copy, 1 - alpha, 0, img_copy)
    return img_copy

# ---------- Main Loop ----------
folder_path = "images"
image_paths = sorted(glob.glob(os.path.join(folder_path, "*.jpg")))
print(f"Processing images: {image_paths}")

for img_path in image_paths:
    img = cv2.imread(img_path)
    img2 = enforce_4_3_aspect_and_scale(img)

    # Extract features
    sub_features, cell_indices = process_image(img2)

    # Normalize features
    sub_features = StandardScaler().fit_transform(sub_features)

    # KMeans clustering
    kmeans = KMeans(n_clusters=NUM_CLUSTERS, random_state=42)
    sub_labels = kmeans.fit_predict(sub_features)

    # Majority vote per grid
    labels = []
    for c in range(NUM_GRIDS):
        cell_subs = sub_labels[cell_indices == c]
        majority_label = np.bincount(cell_subs).argmax()
        labels.append(majority_label)
    labels = np.array(labels)

    # Visualize highlighted grids
    img_highlight = highlight_cells(img2, labels)
    plt.figure(figsize=(10, 8))
    plt.imshow(cv2.cvtColor(img_highlight, cv2.COLOR_BGR2RGB))
    plt.title(f"Highlighted grids: {os.path.basename(img_path)}")
    plt.axis('off')
    plt.show()
    # Save the highlighted image
    save_dir = "output_highlighted"
    os.makedirs(save_dir, exist_ok=True)  # create folder if not exists

    save_path = os.path.join(save_dir, os.path.basename(img_path))
    cv2.imwrite(save_path, img_highlight)

    print(f"Saved processed image to: {save_path}")


In [ ]:
'''
Clusters all 8×8 grid regions from all images together (globally) using multiple texture and color features (Sobel, Laplacian, LBP, Gabor, HSV, etc.), then highlights which regions likely contain animals (non-background)
'''
import cv2
import numpy as np
import glob
import os
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from scipy.stats import skew
from skimage.feature import local_binary_pattern

# ---------- Parameters ----------
GRID_ROWS = 8
GRID_COLS = 8
NUM_GRIDS = GRID_ROWS * GRID_COLS
TARGET_W, TARGET_H = 800, 600
NUM_CLUSTERS = 3   # try 2–4
GABOR_THETAS = np.linspace(0, np.pi, 8, endpoint=False)
GABOR_SIGMAS = [1, 2]
GABOR_LAMBDAS = [4, 8]
GABOR_GAMMAS = [0.5, 1]

# ---------- Utility functions ----------
def enforce_4_3_aspect_and_scale(img):
    h, w = img.shape[:2]
    desired_ratio = 4/3
    cur_ratio = w / h
    if abs(cur_ratio - desired_ratio) > 1e-3:
        if cur_ratio > desired_ratio:
            new_w = int(h * desired_ratio)
            x0 = (w - new_w) // 2
            img = img[:, x0:x0 + new_w]
        else:
            new_h = int(w / desired_ratio)
            y0 = (h - new_h) // 2
            img = img[y0:y0 + new_h, :]
    h2, w2 = img.shape[:2]
    scale = min(1.0, min(TARGET_W / w2, TARGET_H / h2))
    if scale < 1.0:
        new_w = int(w2 * scale)
        new_h = int(h2 * scale)
        img = cv2.resize(img, (new_w, new_h), interpolation=cv2.INTER_AREA)
    pad_w = TARGET_W - img.shape[1]
    pad_h = TARGET_H - img.shape[0]
    top = pad_h // 2 if pad_h > 0 else 0
    bottom = pad_h - top if pad_h > 0 else 0
    left = pad_w // 2 if pad_w > 0 else 0
    right = pad_w - left if pad_w > 0 else 0
    if pad_w > 0 or pad_h > 0:
        img = cv2.copyMakeBorder(img, top, bottom, left, right, cv2.BORDER_REFLECT)
    return img


def split_to_grids(img, rows=GRID_ROWS, cols=GRID_COLS):
    h, w = img.shape[:2]
    cell_h, cell_w = h // rows, w // cols
    grids = []
    for r in range(rows):
        for c in range(cols):
            y0, y1 = r * cell_h, (r + 1) * cell_h
            x0, x1 = c * cell_w, (c + 1) * cell_w
            grids.append(img[y0:y1, x0:x1])
    return grids


# ---------- Feature extraction ----------
def extract_selected_features(cell):
    """
    Extract Sobel, Laplacian, LBP, Gabor, HSV, color ratio, vegetation features.
    """
    gray = cv2.cvtColor(cell, cv2.COLOR_BGR2GRAY)
    hsv = cv2.cvtColor(cell, cv2.COLOR_BGR2HSV)
    b, g, r = cv2.split(cell.astype(np.float32) + 1e-6)
    feats = []

    # --- Sobel ---
    sobelx = cv2.Sobel(gray, cv2.CV_32F, 1, 0, ksize=3)
    sobely = cv2.Sobel(gray, cv2.CV_32F, 0, 1, ksize=3)
    sobel_mag = np.sqrt(sobelx**2 + sobely**2)
    feats += [sobelx.mean(), sobely.mean(), sobel_mag.mean(),
              sobelx.var(), sobely.var(), sobel_mag.var()]

    # --- Laplacian ---
    lap = cv2.Laplacian(gray, cv2.CV_32F)
    feats += [lap.mean(), lap.var(), skew(lap.ravel())]

    # --- LBP ---
    lbp = local_binary_pattern(gray, P=8, R=1, method='uniform')
    (hist, _) = np.histogram(lbp.ravel(), bins=np.arange(0, 10), range=(0, 9))
    hist = hist.astype("float")
    hist /= (hist.sum() + 1e-6)
    feats += hist.tolist()

    # --- Gabor features ---
    for theta in GABOR_THETAS:
        for sigma in GABOR_SIGMAS:
            for lam in GABOR_LAMBDAS:
                for gamma in GABOR_GAMMAS:
                    kernel = cv2.getGaborKernel((9, 9), sigma, theta, lam, gamma, 0, ktype=cv2.CV_32F)
                    filtered = cv2.filter2D(gray, cv2.CV_32F, kernel)
                    feats += [filtered.mean(), filtered.var()]

    # --- HSV stats ---
    h_mean, s_mean, v_mean = hsv[..., 0].mean(), hsv[..., 1].mean(), hsv[..., 2].mean()
    h_std, s_std, v_std = hsv[..., 0].std(), hsv[..., 1].std(), hsv[..., 2].std()
    feats += [h_mean, s_mean, v_mean, h_std, s_std, v_std]

    # --- Color ratios ---
    feats += [(r / g).mean(), (r / b).mean(), (g / b).mean()]

    # --- Vegetation (green) score ---
    H, S, V = hsv[..., 0].astype(np.float32), hsv[..., 1].astype(np.float32), hsv[..., 2].astype(np.float32)
    green_mask = ((H >= 35) & (H <= 100) & (S >= 40) & (V >= 40)).astype(np.uint8)
    veg_score = green_mask.mean()
    feats.append(veg_score)

    return np.array(feats, dtype=np.float32)


def highlight_cells(img, labels, alpha=0.4):
    h, w = img.shape[:2]
    cell_h, cell_w = h // GRID_ROWS, w // GRID_COLS
    img_copy = img.copy()
    for i in range(GRID_ROWS):
        for j in range(GRID_COLS):
            idx = i * GRID_COLS + j
            if labels[idx] == 1:
                y0, y1 = i * cell_h, (i + 1) * cell_h
                x0, x1 = j * cell_w, (j + 1) * cell_w
                overlay = img_copy.copy()
                overlay[y0:y1, x0:x1] = (0, 0, 255)
                cv2.addWeighted(overlay, alpha, img_copy, 1 - alpha, 0, img_copy)
    return img_copy


# ---------- Main pipeline ----------
folder_path = "images"
image_paths = sorted(glob.glob(os.path.join(folder_path, "*.jpg")))
print(f"Found {len(image_paths)} images")

# --- Step 1: Extract features for all images ---
all_features = []
image_to_cells = []
images = []

for img_path in image_paths:
    img = cv2.imread(img_path)
    img2 = enforce_4_3_aspect_and_scale(img)
    images.append((os.path.basename(img_path), img2))

    grids = split_to_grids(img2)
    cell_feats = [extract_selected_features(cell) for cell in grids]

    start_idx = len(all_features)
    all_features.extend(cell_feats)
    end_idx = len(all_features)
    image_to_cells.append((start_idx, end_idx))

# --- Step 2: Train global KMeans ---
print("Training KMeans on all cells...")
scaler = StandardScaler()
X = scaler.fit_transform(all_features)

kmeans = KMeans(n_clusters=NUM_CLUSTERS, random_state=42, n_init=20)
kmeans.fit(X)
print("KMeans trained globally.")

# --- Step 3: Predict and visualize per image ---
save_dir = "output_highlighted"
os.makedirs(save_dir, exist_ok=True)

for (name, img), (start, end) in zip(images, image_to_cells):
    X_cells = X[start:end]
    labels = kmeans.predict(X_cells)

    # Optional heuristic: label cluster with highest mean veg_score as background
    veg_idx = -1
    cluster_mean_veg = [X_cells[labels == k, veg_idx].mean() if np.any(labels == k) else 0 for k in range(NUM_CLUSTERS)]
    bg_cluster = int(np.argmax(cluster_mean_veg))
    labels_binary = (labels != bg_cluster).astype(np.uint8)

    img_highlight = highlight_cells(img, labels_binary)
    plt.figure(figsize=(10, 8))
    plt.imshow(cv2.cvtColor(img_highlight, cv2.COLOR_BGR2RGB))
    plt.title(f"Clusters for {name}")
    plt.axis('off')
    plt.show()

    out_path = os.path.join(save_dir, name)
    cv2.imwrite(out_path, img_highlight)
    print(f"Saved: {out_path}")


In [ ]:
import cv2
import numpy as np
from skimage.feature import hog, local_binary_pattern
from sklearn.cluster import AgglomerativeClustering
import matplotlib.pyplot as plt
import glob, os

# ---------- Parameters ----------
GRID_ROWS = 8
GRID_COLS = 8
NUM_GRIDS = GRID_ROWS * GRID_COLS
TARGET_W, TARGET_H = 800, 600
SUBGRID_SIZE = 4
LBP_PTS = 8
LBP_RADIUS = 1
HOG_PIXELS_PER_CELL = (8, 8)
NUM_CLUSTERS = 2

# ---------- Preprocessing ----------
def enforce_4_3_aspect_and_scale(img):
    h, w = img.shape[:2]
    desired_ratio = 4/3
    cur_ratio = w / h
    if abs(cur_ratio - desired_ratio) > 1e-3:
        if cur_ratio > desired_ratio:
            new_w = int(h * desired_ratio)
            x0 = (w - new_w) // 2
            img = img[:, x0:x0+new_w]
        else:
            new_h = int(w / desired_ratio)
            y0 = (h - new_h) // 2
            img = img[y0:y0+new_h, :]
    img = cv2.resize(img, (TARGET_W, TARGET_H), interpolation=cv2.INTER_AREA)
    return img

def split_to_grids(img, rows=GRID_ROWS, cols=GRID_COLS):
    h, w = img.shape[:2]
    cell_h, cell_w = h // rows, w // cols
    grids = []
    for r in range(rows):
        for c in range(cols):
            y0, x0 = r*cell_h, c*cell_w
            grids.append(img[y0:y0+cell_h, x0:x0+cell_w])
    return grids

def extract_features_for_cell(cell):
    gray = cv2.cvtColor(cell, cv2.COLOR_BGR2GRAY)
    hog_feat = hog(gray, pixels_per_cell=HOG_PIXELS_PER_CELL, cells_per_block=(2,2), feature_vector=True)
    lbp = local_binary_pattern(gray, LBP_PTS, LBP_RADIUS, method='uniform')
    hist_lbp, _ = np.histogram(lbp.ravel(), bins=np.arange(0, LBP_PTS+3), range=(0, LBP_PTS+2))
    hist_lbp = hist_lbp / (hist_lbp.sum() + 1e-6)
    hsv = cv2.cvtColor(cell, cv2.COLOR_BGR2HSV)
    h_hist = cv2.calcHist([hsv],[0],None,[16],[0,180]).flatten()
    s_hist = cv2.calcHist([hsv],[1],None,[8],[0,256]).flatten()
    color_hist = np.concatenate([h_hist,s_hist]) / (h_hist.sum()+s_hist.sum()+1e-6)
    gabor_feats = []
    for theta in [0, np.pi/4, np.pi/2, 3*np.pi/4]:
        kernel = cv2.getGaborKernel((7,7),3,theta,10,0.5,0)
        filtered = cv2.filter2D(gray, cv2.CV_64F, kernel)
        gabor_feats.extend([filtered.mean(), filtered.std()])
    gabor_feats = np.array(gabor_feats)
    feat = np.concatenate([hog_feat, hist_lbp, color_hist, gabor_feats])
    return feat

def extract_features_for_subblocks(cell):
    h, w = cell.shape[:2]
    sub_h, sub_w = h//SUBGRID_SIZE, w//SUBGRID_SIZE
    feats = []
    for i in range(SUBGRID_SIZE):
        for j in range(SUBGRID_SIZE):
            sub = cell[i*sub_h:(i+1)*sub_h, j*sub_w:(j+1)*sub_w]
            feats.append(extract_features_for_cell(sub))
    return np.array(feats)

def process_image(img):
    grids = split_to_grids(img)
    all_feats, cell_ids = [], []
    for idx, cell in enumerate(grids):
        sub_feats = extract_features_for_subblocks(cell)
        all_feats.extend(sub_feats)
        cell_ids.extend([idx]*len(sub_feats))
    return np.array(all_feats), np.array(cell_ids)

def highlight_cells(img, labels, alpha=0.4):
    h, w = img.shape[:2]
    cell_h, cell_w = h // GRID_ROWS, w // GRID_COLS
    img_copy = img.copy()
    for i in range(GRID_ROWS):
        for j in range(GRID_COLS):
            idx = i * GRID_COLS + j
            if labels[idx] == 1:
                y0, y1 = i * cell_h, (i + 1) * cell_h
                x0, x1 = j * cell_w, (j + 1) * cell_w
                overlay = img_copy.copy()
                overlay[y0:y1, x0:x1] = (0, 0, 255)
                cv2.addWeighted(overlay, alpha, img_copy, 1 - alpha, 0, img_copy)
    return img_copy

# ---------- MAIN (Hierarchical clustering) ----------
folder_path = "images"
image_paths = sorted(glob.glob(folder_path + "/*.jpg"))

for path in image_paths:
    img = cv2.imread(path)
    img = enforce_4_3_aspect_and_scale(img)
    feats, cell_ids = process_image(img)
    
    model = AgglomerativeClustering(n_clusters=NUM_CLUSTERS, linkage='ward')
    sub_labels = model.fit_predict(feats)
    
    labels = []
    for c in range(NUM_GRIDS):
        sub = sub_labels[cell_ids==c]
        if len(sub)>0:
            labels.append(np.bincount(sub).argmax())
        else:
            labels.append(0)
    labels = np.array(labels)
    
    highlighted = highlight_cells(img, labels)
    plt.figure(figsize=(10,8))
    plt.imshow(cv2.cvtColor(highlighted, cv2.COLOR_BGR2RGB))
    plt.title(f"Hierarchical clustering: {os.path.basename(path)}")
    plt.axis('off')
    plt.show()


In [ ]:
from scipy.spatial.distance import cdist

def region_growing_segmentation(features, rows=GRID_ROWS, cols=GRID_COLS, threshold=0.3):
    """Simple region growing based on mean feature distance between adjacent grids."""
    labels = -np.ones(rows*cols, dtype=int)
    cur_label = 0
    
    def get_neighbors(idx):
        r, c = divmod(idx, cols)
        nbs = []
        for dr, dc in [(-1,0),(1,0),(0,-1),(0,1)]:
            nr, nc = r+dr, c+dc
            if 0 <= nr < rows and 0 <= nc < cols:
                nbs.append(nr*cols + nc)
        return nbs

    for idx in range(rows*cols):
        if labels[idx] != -1:
            continue
        labels[idx] = cur_label
        region = [idx]
        queue = [idx]
        region_mean = features[idx]
        
        while queue:
            current = queue.pop(0)
            for nb in get_neighbors(current):
                if labels[nb] == -1:
                    dist = np.linalg.norm(features[nb] - region_mean)
                    if dist < threshold:
                        labels[nb] = cur_label
                        queue.append(nb)
                        region.append(nb)
                        region_mean = features[region].mean(axis=0)
        cur_label += 1
    return labels

# ---------- Example use ----------
for path in image_paths:
    img = cv2.imread(path)
    img = enforce_4_3_aspect_and_scale(img)
    
    grids = split_to_grids(img)
    features = np.array([extract_features_for_cell(g) for g in grids])
    
    labels = region_growing_segmentation(features, GRID_ROWS, GRID_COLS, threshold=0.4)
    
    highlighted = highlight_cells(img, labels % 2)  # visualize 2 colors
    plt.figure(figsize=(10,8))
    plt.imshow(cv2.cvtColor(highlighted, cv2.COLOR_BGR2RGB))
    plt.title(f"Region Growing: {os.path.basename(path)}")
    plt.axis('off')
    plt.show()
